<a href="https://colab.research.google.com/github/ayyucedemirbas/WSI_to_ST/blob/main/WSI_to_ST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import pandas as pd
from huggingface_hub import snapshot_download

meta_df = pd.read_csv("hf://datasets/MahmoodLab/hest/HEST_v1_3_0.csv")

filtered_df = meta_df[meta_df['organ'] == 'Breast'].head(5)
ids_to_query = filtered_df['id'].values
print(f"Patient IDs: {ids_to_query}")

list_patterns = [f"*{patient_id}[_.]*" for patient_id in ids_to_query]

snapshot_download(
    repo_id='MahmoodLab/hest',
    repo_type="dataset",
    allow_patterns=list_patterns,
    local_dir='/content/hest_data'
)


Patient IDs: ['TENX202' 'TENX201' 'TENX200' 'TENX199' 'TENX198']


Fetching ... files: 0it [00:00, ?it/s]

'/content/hest_data'

In [4]:
!pip install PeekDir

In [5]:
!PeekDir hest_data

hest_data/
    .cache/
        huggingface/
            download/
                cellvit_seg/
                    TENX198_cellvit_seg.geojson.zip.metadata
                    TENX198_cellvit_seg.parquet.metadata
                    TENX199_cellvit_seg.geojson.zip.metadata
                    TENX199_cellvit_seg.parquet.metadata
                    TENX200_cellvit_seg.geojson.zip.metadata
                    ... 5 more .metadata files
                metadata/
                    TENX198.json.metadata
                    TENX199.json.metadata
                    TENX200.json.metadata
                    TENX201.json.metadata
                    TENX202.json.metadata
                patches/
                    TENX198.h5.metadata
                    TENX199.h5.metadata
                    TENX200.h5.metadata
                    TENX201.h5.metadata
                    TENX202.h5.metadata
                patches_vis/
                    TENX198.png.metadata
                    TENX199.pn

In [7]:
!pip install imagecodecs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 55.0 MB/s eta 0:00:00


In [3]:
!pip install openslide-python

In [4]:
!pip install openslide-bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 61.3 MB/s eta 0:00:00


In [1]:
from __future__ import annotations

import gc
import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Protocol, Tuple

import cv2
import numpy as np
from PIL import Image
from scipy import linalg
from tqdm import tqdm

warnings.filterwarnings("ignore")

def _bootstrap_openslide() -> bool:
    try:
        import openslide_bin
    except ImportError:
        pass
    try:
        import openslide
        return True
    except ImportError:
        return False

_OPENSLIDE_OK = _bootstrap_openslide()

class SlideReader(Protocol):
    @property
    def level_count(self) -> int: ...
    @property
    def level_dimensions(self) -> list[tuple[int, int]]: ...
    @property
    def level_downsamples(self) -> list[float]: ...
    def get_thumbnail(self, size: tuple[int, int]) -> Image.Image: ...
    def read_region(self, location: tuple[int, int], level: int, size: tuple[int, int]) -> Image.Image: ...
    def close(self) -> None: ...
    @property
    def properties(self) -> dict: ...

class OpenSlideReader:
    def __init__(self, path: str):
        import openslide
        self._slide = openslide.OpenSlide(path)

    @property
    def level_count(self):
        return self._slide.level_count

    @property
    def level_dimensions(self):
        return self._slide.level_dimensions

    @property
    def level_downsamples(self):
        return self._slide.level_downsamples

    @property
    def properties(self):
        return self._slide.properties

    def get_thumbnail(self, size: tuple[int, int]) -> Image.Image:
        return self._slide.get_thumbnail(size)

    def read_region(self, location: tuple[int, int], level: int, size: tuple[int, int]) -> Image.Image:
        return self._slide.read_region(location, level, size)

    def close(self):
        self._slide.close()

class TifffileReader:
    def __init__(self, path: str):
        import tifffile
        self._tif = tifffile.TiffFile(path)
        self._path = path
        self._pages = [p for p in self._tif.pages if len(p.shape) == 3 and p.shape[2] in (3, 4)]
        if not self._pages:
            raise ValueError("tifffile: no RGB pages found in the WSI.")
        self._pages.sort(key=lambda p: p.shape[0] * p.shape[1], reverse=True)
        W0, H0 = self._pages[0].shape[1], self._pages[0].shape[0]
        self._dims = [(p.shape[1], p.shape[0]) for p in self._pages]
        self._dsamp = [W0 / d[0] for d in self._dims]

    @property
    def level_count(self):
        return len(self._pages)

    @property
    def level_dimensions(self):
        return self._dims

    @property
    def level_downsamples(self):
        return self._dsamp

    @property
    def properties(self):
        return {}

    def get_thumbnail(self, size: tuple[int, int]) -> Image.Image:
        small = self._pages[-1].asarray()
        if small.shape[2] == 4:
            small = small[:, :, :3]
        img = Image.fromarray(small, "RGB")
        img.thumbnail(size, Image.LANCZOS)
        return img

    def read_region(self, location: tuple[int, int], level: int, size: tuple[int, int]) -> Image.Image:
        level = min(level, self.level_count - 1)
        ds = self._dsamp[level]
        x_lv = int(location[0] / ds)
        y_lv = int(location[1] / ds)
        W_lv, H_lv = self._dims[level]
        pw, ph = size
        x1 = min(x_lv, W_lv)
        y1 = min(y_lv, H_lv)
        x2 = min(x_lv + pw, W_lv)
        y2 = min(y_lv + ph, H_lv)
        try:
            region = self._pages[level].asarray()[y1:y2, x1:x2]
        except Exception:
            full = self._pages[level].asarray()
            region = full[y1:y2, x1:x2]
        if region.shape[2] == 4:
            region = region[:, :, :3]
        img = Image.fromarray(region, "RGB")
        if img.size != (pw, ph):
            padded = Image.new("RGB", (pw, ph), (255, 255, 255))
            padded.paste(img, (0, 0))
            img = padded
        return img

    def close(self):
        self._tif.close()

def open_slide(path: str) -> SlideReader:
    if _OPENSLIDE_OK:
        try:
            reader = OpenSlideReader(path)
            return reader
        except Exception:
            pass
    try:
        import tifffile
        reader = TifffileReader(path)
        return reader
    except ImportError:
        raise RuntimeError("Neither openslide nor tifffile can open the file.")
    except Exception as e:
        raise RuntimeError(f"tifffile also failed: {e}")

@dataclass
class PipelineConfig:
    wsi_path: str = "slide.svs"
    output_dir: str = "output_patches"
    mask_level: int = 2
    otsu_channel: str = "saturation"
    tissue_threshold: float = 0.5
    patch_level: int = 0
    patch_size: int = 256
    stride: int = 256
    normalize: bool = True
    reference_patch_path: Optional[str] = None
    macenko_percentile: float = 99.0
    macenko_beta: float = 0.15
    save_mask: bool = True
    save_format: str = "png"
    max_patches: Optional[int] = None

class TissueMasker:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

    def build_mask(self, slide: SlideReader) -> tuple[np.ndarray, tuple[int, int]]:
        level = min(self.cfg.mask_level, slide.level_count - 1)
        thumb_size = slide.level_dimensions[level]
        thumbnail = slide.get_thumbnail(thumb_size)
        thumb_np = np.array(thumbnail.convert("RGB"))
        if self.cfg.otsu_channel == "saturation":
            hsv = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2HSV)
            channel = hsv[:, :, 1]
            _, mask = cv2.threshold(channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            gray = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2GRAY)
            _, mask = cv2.threshold(255 - gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
        return mask, thumb_size

    def save_mask(self, mask: np.ndarray, out_dir: Path) -> None:
        p = out_dir / "tissue_mask.png"
        Image.fromarray(mask).save(str(p))

class MacenkoNormalizer:
    def __init__(self, beta: float = 0.15, percentile: float = 99.0):
        self.beta = beta
        self.percentile = percentile
        self._stain_matrix_ref: Optional[np.ndarray] = None
        self._max_conc_ref: Optional[np.ndarray] = None

    @staticmethod
    def _rgb_to_od(img: np.ndarray) -> np.ndarray:
        return -np.log(np.maximum(img.astype(np.float64), 1) / 255.0)

    @staticmethod
    def _od_to_rgb(od: np.ndarray) -> np.ndarray:
        return np.clip(np.exp(-od) * 255, 0, 255).astype(np.uint8)

    def _get_stain_matrix(self, img_rgb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        od = self._rgb_to_od(img_rgb).reshape(-1, 3)
        od = od[np.linalg.norm(od, axis=1) > self.beta]
        if od.shape[0] < 10:
            return (np.array([[0.6500, 0.7042, 0.2860], [0.0704, 0.9911, 0.1120]]), np.array([1.0, 1.0]))
        _, _, Vt = linalg.svd(od, full_matrices=False)
        plane = Vt[:2].T
        proj = od @ plane
        angles = np.arctan2(proj[:, 1], proj[:, 0])
        phi_min = np.percentile(angles, 100 - self.percentile)
        phi_max = np.percentile(angles, self.percentile)
        v1 = plane @ np.array([np.cos(phi_min), np.sin(phi_min)])
        v2 = plane @ np.array([np.cos(phi_max), np.sin(phi_max)])
        v1 /= np.linalg.norm(v1) + 1e-6
        v2 /= np.linalg.norm(v2) + 1e-6
        if v1[0] < v2[0]:
            v1, v2 = v2, v1
        stain = np.stack([v1, v2])
        conc = od @ np.linalg.pinv(stain)
        max_conc = np.percentile(conc, self.percentile, axis=0)
        return stain, max_conc

    def fit(self, reference_rgb: np.ndarray) -> "MacenkoNormalizer":
        self._stain_matrix_ref, self._max_conc_ref = self._get_stain_matrix(reference_rgb)
        return self

    def transform(self, img_rgb: np.ndarray) -> np.ndarray:
        if self._stain_matrix_ref is None:
            raise RuntimeError("Call .fit() before .transform().")
        H, W = img_rgb.shape[:2]
        od = self._rgb_to_od(img_rgb).reshape(-1, 3)
        stain_src, max_conc_src = self._get_stain_matrix(img_rgb)
        conc = od @ np.linalg.pinv(stain_src)
        conc_norm = conc / (max_conc_src + 1e-6) * self._max_conc_ref
        od_norm = conc_norm @ self._stain_matrix_ref
        return self._od_to_rgb(od_norm).reshape(H, W, 3)

class WSIPatcher:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

    def _patch_has_tissue(self, mask_l0: np.ndarray, x: int, y: int, patch_size_l0: int) -> bool:
        region = mask_l0[y: y + patch_size_l0, x: x + patch_size_l0]
        return region.size > 0 and (region > 0).mean() >= self.cfg.tissue_threshold

    def extract_and_save(self, slide: SlideReader, mask: np.ndarray, thumb_size: tuple[int, int], normalizer: Optional[MacenkoNormalizer], out_dir: Path) -> int:
        patch_level = min(self.cfg.patch_level, slide.level_count - 1)
        ds = slide.level_downsamples[patch_level]
        W_pl, H_pl = slide.level_dimensions[patch_level]
        W0, H0 = slide.level_dimensions[0]
        ps = self.cfg.patch_size
        stride = self.cfg.stride
        patch_size_l0 = int(ps * ds)
        mask_l0 = cv2.resize(mask, (W0, H0), interpolation=cv2.INTER_NEAREST)
        cols = (W_pl - ps) // stride + 1
        rows = (H_pl - ps) // stride + 1
        patches_dir = out_dir / "patches"
        patches_dir.mkdir(parents=True, exist_ok=True)
        saved = 0
        with tqdm(total=cols * rows, desc="Patching", unit="patch") as pbar:
            for row in range(rows):
                for col in range(cols):
                    x_pl = col * stride
                    y_pl = row * stride
                    x_l0 = int(x_pl * ds)
                    y_l0 = int(y_pl * ds)
                    pbar.update(1)
                    if not self._patch_has_tissue(mask_l0, x_l0, y_l0, patch_size_l0):
                        continue
                    region = slide.read_region((x_l0, y_l0), patch_level, (ps, ps))
                    patch_rgb = np.array(region.convert("RGB"))
                    if normalizer is not None:
                        try:
                            patch_rgb = normalizer.transform(patch_rgb)
                        except Exception:
                            pass
                    fname = f"patch_r{row:05d}_c{col:05d}.{self.cfg.save_format}"
                    Image.fromarray(patch_rgb).save(str(patches_dir / fname))
                    saved += 1
                    if self.cfg.max_patches and saved >= self.cfg.max_patches:
                        pbar.close()
                        break
                    if saved % 500 == 0:
                        gc.collect()
                if self.cfg.max_patches and saved >= self.cfg.max_patches:
                    break
        return saved

class WSIPipeline:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.out_dir = Path(cfg.output_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def _auto_reference(self, slide: SlideReader, mask: np.ndarray, thumb_size: tuple, n_max: int = 30) -> Optional[np.ndarray]:
        patcher = WSIPatcher(self.cfg)
        pl = min(self.cfg.patch_level, slide.level_count - 1)
        ds = slide.level_downsamples[pl]
        W_pl, H_pl = slide.level_dimensions[pl]
        W0, H0 = slide.level_dimensions[0]
        ps = self.cfg.patch_size
        ps_l0 = int(ps * ds)
        mask_l0 = cv2.resize(mask, (W0, H0), interpolation=cv2.INTER_NEAREST)
        found = 0
        for y in range(0, H_pl - ps, ps):
            for x in range(0, W_pl - ps, ps):
                x0, y0 = int(x * ds), int(y * ds)
                if not patcher._patch_has_tissue(mask_l0, x0, y0, ps_l0):
                    continue
                region = slide.read_region((x0, y0), pl, (ps, ps))
                return np.array(region.convert("RGB"))
            found += 1
            if found >= n_max:
                break
        return None

    def run(self) -> None:
        slide = open_slide(self.cfg.wsi_path)
        masker = TissueMasker(self.cfg)
        mask, thumb_size = masker.build_mask(slide)
        if self.cfg.save_mask:
            masker.save_mask(mask, self.out_dir)
        normalizer: Optional[MacenkoNormalizer] = None
        if self.cfg.normalize:
            if self.cfg.reference_patch_path:
                ref_rgb = np.array(Image.open(self.cfg.reference_patch_path).convert("RGB"))
            else:
                ref_rgb = self._auto_reference(slide, mask, thumb_size)
            if ref_rgb is not None:
                normalizer = MacenkoNormalizer(beta=self.cfg.macenko_beta, percentile=self.cfg.macenko_percentile).fit(ref_rgb)
        patcher = WSIPatcher(self.cfg)
        patcher.extract_and_save(slide, mask, thumb_size, normalizer, self.out_dir)
        slide.close()

def main():
    wsi_dir = Path("hest_data/wsis")
    wsi_files = list(wsi_dir.glob("*.tif"))

    for wsi_path in wsi_files:
        patient_id = wsi_path.stem
        out_dir = f"wsi_patches_output/{patient_id}"

        print(f"[{patient_id}] is processing...")
        cfg = PipelineConfig(
            wsi_path=str(wsi_path),
            output_dir=out_dir,
            mask_level=2,
            otsu_channel="saturation",
            patch_level=0,
            patch_size=256,
            stride=256,
            tissue_threshold=0.5,
            normalize=True,
            reference_patch_path=None,
            macenko_percentile=99.0,
            macenko_beta=0.15,
            max_patches=None,
            save_mask=True,
            save_format="png",
        )
        pipeline = WSIPipeline(cfg)
        pipeline.run()

if __name__ == "__main__":
    main()

[TENX198] is processing...


Patching: 100%|██████████| 17526/17526 [08:30<00:00, 34.36patch/s] 


[TENX201] is processing...


Patching: 100%|██████████| 13706/13706 [05:20<00:00, 42.74patch/s] 


[TENX200] is processing...


Patching: 100%|██████████| 18824/18824 [04:27<00:00, 70.46patch/s] 


[TENX199] is processing...


Patching: 100%|██████████| 7134/7134 [02:29<00:00, 47.69patch/s] 


[TENX202] is processing...


Patching: 100%|██████████| 9912/9912 [04:47<00:00, 34.44patch/s] 


In [3]:
!pip install anndata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.8 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import anndata as ad
from pathlib import Path
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class SpatialTranscriptomicsDataset(Dataset):
    def __init__(self, patient_ids, patches_dir_base, st_dir_base, transform=None, num_genes=250):
        self.transform = transform
        self.num_genes = num_genes
        self.samples = []

        print(f"Preparing dataset... Patients: {patient_ids}")
        for pid in patient_ids:
            st_path = Path(st_dir_base) / f"{pid}.h5ad"
            if not st_path.exists():
                continue

            adata = ad.read_h5ad(st_path)

            expressions = adata.X[:, :num_genes]
            if hasattr(expressions, "toarray"):
                expressions = expressions.toarray()

            patient_patches_dir = Path(patches_dir_base) / pid / "patches"
            if not patient_patches_dir.exists():
                continue

            patch_files = list(patient_patches_dir.glob("*.png"))

            num_available_spots = expressions.shape[0]
            for i, patch_path in enumerate(patch_files):
                if i >= num_available_spots:
                    break

                target_expression = expressions[i]
                self.samples.append((str(patch_path), target_expression))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, expression = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(expression, dtype=torch.float32)
        return image, target

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

all_patients = ["TENX198", "TENX199", "TENX200", "TENX201", "TENX202"]

train_ids = all_patients[:3]
val_ids   = [all_patients[3]]
test_ids  = [all_patients[4]]

NUM_GENES_TO_PREDICT = 250

train_dataset = SpatialTranscriptomicsDataset(train_ids, "wsi_patches_output", "hest_data/st", transform=train_transform, num_genes=NUM_GENES_TO_PREDICT)
val_dataset   = SpatialTranscriptomicsDataset(val_ids, "wsi_patches_output", "hest_data/st", transform=val_test_transform, num_genes=NUM_GENES_TO_PREDICT)
test_dataset  = SpatialTranscriptomicsDataset(test_ids, "wsi_patches_output", "hest_data/st", transform=val_test_transform, num_genes=NUM_GENES_TO_PREDICT)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Total Patches -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

class WSI2STModel(nn.Module):
    def __init__(self, num_genes):
        super(WSI2STModel, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_genes)
        )

    def forward(self, x):
        return self.backbone(x)

model = WSI2STModel(num_genes=NUM_GENES_TO_PREDICT).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

EPOCHS = 10

print("Training starting...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss (MSE): {train_loss:.4f} | Val Loss (MSE): {val_loss:.4f}")

torch.save(model.state_dict(), "wsi2st_resnet18.pth")
print("Model successfully saved!")

Using device: cuda
Preparing dataset... Patients: ['TENX198', 'TENX199', 'TENX200']
Preparing dataset... Patients: ['TENX201']
Preparing dataset... Patients: ['TENX202']
Total Patches -> Train: 18002, Val: 6993, Test: 5599
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


Training starting...


Epoch 1/10 [Val]: 100%|██████████| 219/219 [00:29<00:00,  7.44it/s]


Epoch 1/10 | Train Loss (MSE): 16304.2414 | Val Loss (MSE): 30988.6544


...

Epoch 8/10 | Train Loss (MSE): 10014.0143 | Val Loss (MSE): 31865.0841


Epoch 9/10 [Val]: 100%|██████████| 219/219 [00:24<00:00,  8.99it/s]


Epoch 9/10 | Train Loss (MSE): 9797.8287 | Val Loss (MSE): 31759.4884


Epoch 10/10 [Val]: 100%|██████████| 219/219 [00:24<00:00,  9.10it/s]

Epoch 10/10 | Train Loss (MSE): 9612.6810 | Val Loss (MSE): 31825.9991
Model successfully saved!


In [5]:
from scipy.stats import pearsonr

model.eval()

test_loss = 0.0
criterion = nn.MSELoss()
all_preds = []
all_targets = []


with torch.no_grad():
    for i, (images, targets) in enumerate(test_loader):
        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * images.size(0)

        all_preds.append(outputs.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

        if i == 0:
            for j in range(min(3, images.size(0))):
                print(f"\nPatch {j+1}:")
                print(f"Predicted (first 5 genes): {outputs[j, :5].cpu().numpy().round(4)}")
                print(f"Actual    (first 5 genes): {targets[j, :5].cpu().numpy().round(4)}")

test_loss /= len(test_loader.dataset)
print(f"Final Test Loss (MSE): {test_loss:.4f}")

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

correlations = []
for g in range(all_preds.shape[1]):
    if np.std(all_targets[:, g]) > 0 and np.std(all_preds[:, g]) > 0:
        corr, _ = pearsonr(all_preds[:, g], all_targets[:, g])
        correlations.append(corr)

if correlations:
    mean_corr = np.mean(correlations)
    print(f"Mean Pearson Correlation across genes: {mean_corr:.4f}")
else:
    print("Could not compute Pearson Correlation.")


Patch 1:
Predicted (first 5 genes): [31.3301 71.8321 19.6507 98.724   7.141 ]
Actual    (first 5 genes): [0. 0. 0. 0. 0.]

Patch 2:
Predicted (first 5 genes): [32.0024 70.6163 22.5921 96.8934  8.24  ]
Actual    (first 5 genes): [0. 0. 0. 0. 0.]

Patch 3:
Predicted (first 5 genes): [31.4064 71.8379 19.869  98.7224  7.2218]
Actual    (first 5 genes): [0. 0. 0. 0. 0.]
Final Test Loss (MSE): 10564.5122
Mean Pearson Correlation across genes: -0.0014
